# Tarea 2

Autor: Diego Prado

In [33]:
import os
import polars as pl
from polars import DataFrame
import plotly.express as px
import plotly.io as pio
from dash_bootstrap_templates import load_figure_template
from path import Path
import json
import numpy as np

#Paquetería para pruebas estadisticas
from scipy.stats import skew, kurtosis

#Paqueterías de limpieza de datos
from sklearn.impute import SimpleImputer

## Confuguraciones

In [2]:
pio.renderers.default = 'iframe'
load_figure_template("minty_dark")

## Funciones

In [3]:
def report_nulls(df: DataFrame) -> DataFrame:
    """
    Generamos un reporte de nulos de los datos.
    """
    cols = df.columns
    report = {
        "columns":[],
        "nulls":[]
    }
    
    for col in cols:
        nulos = df[col].is_null().sum()
        if nulos > 0:
            report.get("columns").append(col)
            report.get("nulls").append(nulos)
    
    report = pl.DataFrame(report)
    report = report.with_columns(
        prc_nulls = pl.col("nulls") / df.shape[0] * 100
    )

    return report


    

In [4]:
def plot_nulls(df: DataFrame) -> None:
    """
    Graficamos los nulos de los dataframes.
    """

    fig = px.bar(
        df, 
        x="prc_nulls", 
        y="columns", 
        title="Porcentaje de Valores Nulos por Columna",
        text_auto=True,
        height=800,
        width=900,
        range_x=[0,100]
    )

    fig.show()

## Limpieza de datos

In [5]:
p = Path('data')

df = pl.read_csv(p / 'raw' / 'FIFA19-DS.csv')
df.head()

ID,Age,Overall,Potential,Value,Wage,Special,International Reputation,Weak Foot,Skill Moves,Work Rate,Height,Weight,LS,ST,RS,LW,LF,CF,RF,RW,LAM,CAM,RAM,LM,LCM,CM,RCM,RM,LWB,LDM,CDM,RDM,RWB,LB,LCB,CB,…,Crossing,Finishing,HeadingAccuracy,ShortPassing,Volleys,Dribbling,Curve,FKAccuracy,LongPassing,BallControl,Acceleration,SprintSpeed,Agility,Reactions,Balance,ShotPower,Jumping,Stamina,Strength,LongShots,Aggression,Interceptions,Positioning,Vision,Penalties,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Preferred Foot,Body Type,Position
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str
158023,31,94,94,110500000,565000,2202,5,4,4,4,5.7,159,90,90,90,94,95,95,95,94,95,95,95,93,86,86,86,93,66,63,63,63,66,61,49,49,…,84,95,70,90,86,97,93,94,87,96,91,86,91,95,95,85,68,72,59,94,48,22,94,94,75,96,33,28,26,6,11,15,14,8,"""Left""","""Lean""","""ST"""
20801,33,94,94,77000000,405000,2228,5,4,5,6,6.2,183,94,94,94,92,93,93,93,92,91,91,91,91,84,84,84,91,68,64,64,64,68,64,56,56,…,84,94,89,81,87,88,81,76,77,94,89,91,87,96,70,95,95,88,79,93,63,29,95,82,85,95,28,31,23,7,11,15,14,11,"""Right""","""Normal""","""ST"""
190871,26,92,93,118500000,290000,2143,5,5,5,7,5.9,150,87,87,87,92,92,92,92,92,92,92,92,91,84,84,84,91,68,63,63,63,68,63,50,50,…,79,87,62,84,84,96,88,87,78,95,94,90,96,94,84,80,61,81,49,82,56,36,89,87,81,94,27,24,33,9,9,15,15,11,"""Right""","""Lean""","""AM"""
192985,27,91,92,102000000,355000,2281,4,5,4,8,5.11,154,85,85,85,90,90,90,90,90,91,91,91,91,90,90,90,91,80,80,80,80,80,76,69,69,…,93,82,55,92,82,86,85,83,91,91,78,76,79,91,77,91,63,90,75,91,76,61,87,94,79,88,68,58,51,15,13,5,10,13,"""Right""","""Normal""","""MF"""
183277,27,91,91,93000000,340000,2142,4,4,4,7,5.8,163,86,86,86,92,91,91,91,92,92,92,92,92,85,85,85,92,69,66,66,66,69,63,52,52,…,81,84,61,89,80,95,83,79,83,94,94,88,95,90,94,82,56,83,66,80,54,41,87,89,86,91,34,27,22,11,12,6,8,8,"""Right""","""Normal""","""ST"""


In [35]:
df.describe()

statistic,ID,Age,Overall,Potential,Value,Wage,Special,International Reputation,Weak Foot,Skill Moves,Work Rate,Height,Weight,LS,ST,RS,LW,LF,CF,RF,RW,LAM,CAM,RAM,LM,LCM,CM,RCM,RM,LWB,LDM,CDM,RDM,RWB,LB,LCB,…,Crossing,Finishing,HeadingAccuracy,ShortPassing,Volleys,Dribbling,Curve,FKAccuracy,LongPassing,BallControl,Acceleration,SprintSpeed,Agility,Reactions,Balance,ShotPower,Jumping,Stamina,Strength,LongShots,Aggression,Interceptions,Positioning,Vision,Penalties,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Preferred Foot,Body Type,Position
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""count""",17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,…,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,17140.0,"""17140""","""17140""","""17140"""
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,1842.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0"""
"""mean""",214049.431855,25.150117,66.371937,71.386873,2.4867e6,10007.526254,1604.909218,1.117095,2.957351,2.375788,4.75846,5.795376,165.956243,60.015492,60.015492,60.015492,61.267224,60.946921,60.946921,60.946921,61.267224,61.223297,61.223297,61.223297,61.964897,60.469342,60.469342,60.469342,61.964897,59.720486,59.041116,59.041116,59.041116,59.720486,58.999346,57.782521,…,50.15811,45.875554,52.51937,59.045041,43.262427,55.765753,47.574037,43.240665,53.084714,58.738798,64.791774,64.892824,63.715811,61.987981,64.105601,55.825846,65.139907,63.495741,65.40105,47.513069,56.179055,47.009918,50.339498,53.744866,48.779055,58.876196,47.538798,48.017853,45.95175,16.399242,16.179872,16.02993,16.173454,16.485414,null,null,null
"""std""",30013.910983,4.658667,6.925482,6.159271,5.7320e6,22511.507231,272.616761,0.400711,0.661593,0.755884,1.607135,0.448955,15.54517,9.131929,9.131929,9.131929,9.869031,9.834071,9.834071,9.834071,9.869031,9.763917,9.763917,9.763917,9.247231,8.818658,8.818658,8.818658,9.247231,8.974868,10.077393,10.077393,10.077393,8.974868,9.525282,11.682026,…,18.291,19.455131,17.23845,14.596055,17.695322,18.785234,18.376953,17.495024,15.260262,16.571686,14.884353,14.596992,14.730155,9.019559,14.105805,17.162725,11.820453,15.839904,12.508391,19.215724,17.285448,20.63322,19.42724,14.128352,15.653902,11.44456,19.835394,21.580262,21.217484,17.408058,16.646432,16.246873,16.752596,17.668488,null,null,null
"""min""",16.0,16.0,46.0,48.0,0.0,0.0,731.0,1.0,1.0,1.0,0.0,5.1,110.0,33.0,33.0,33.0,27.0,29.0,29.0,29.0,27.0,29.0,29.0,29.0,29.0,32.0,32.0,32.0,29.0,32.0,30.0,30.0,30.0,32.0,31.0,27.0,…,5.0,2.0,4.0,7.0,4.0,4.0,6.0,4.0,9.0,8.0,12.0,12.0,14.0,21.0,16.0,2.0,15.0,12.0,17.0,3.0,11.0,3.0,2.0,10.0,5.0,12.0,3.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,"""Left""","""Lean""","""AM"""
"""25%""",199940.0,21.0,62.0,67.0,325000.0,1000.0,1465.0,1.0,3.0,2.0,4.0,5.11,154.0,54.0,54.0,54.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,56.0,55.0,55.0,55.0,56.0,54.0,52.0,52.0,52.0,54.0,52.0,48.0,…,39.0,30.0,45.0,54.0,31.0,49.0,34.0,31.0,44.0,54.0,57.0,58.0,56.0,56.0,56.0,46.0,58.0,56.0,58.0,33.0,44.0,26.0,3

In [6]:
df.is_duplicated().sum()

0

In [7]:
plot_nulls(report_nulls(df))

In [12]:
fig = px.bar(
    df.group_by("Position").agg(pl.col('ID').count()),
    x="Position",
    y="ID",
    title="Total de Jugadores por posición",
    text_auto=True
)

fig.show()

In [45]:
null_columns = report_nulls(df)["columns"].to_list()

df.group_by("Position").agg(
    [pl.col(c).is_null().sum().alias(f"{c}_nulls") for c in null_columns]
)

Position,LS_nulls,ST_nulls,RS_nulls,LW_nulls,LF_nulls,CF_nulls,RF_nulls,RW_nulls,LAM_nulls,CAM_nulls,RAM_nulls,LM_nulls,LCM_nulls,CM_nulls,RCM_nulls,RM_nulls,LWB_nulls,LDM_nulls,CDM_nulls,RDM_nulls,RWB_nulls,LB_nulls,LCB_nulls,CB_nulls,RCB_nulls,RB_nulls
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""DF""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""ST""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""AM""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""GK""",1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842,1842
"""DM""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""MF""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [49]:
df_sin_porteros = df.filter(pl.col("Position") != "GK")
df_sin_porteros.describe()

statistic,ID,Age,Overall,Potential,Value,Wage,Special,International Reputation,Weak Foot,Skill Moves,Work Rate,Height,Weight,LS,ST,RS,LW,LF,CF,RF,RW,LAM,CAM,RAM,LM,LCM,CM,RCM,RM,LWB,LDM,CDM,RDM,RWB,LB,LCB,…,Crossing,Finishing,HeadingAccuracy,ShortPassing,Volleys,Dribbling,Curve,FKAccuracy,LongPassing,BallControl,Acceleration,SprintSpeed,Agility,Reactions,Balance,ShotPower,Jumping,Stamina,Strength,LongShots,Aggression,Interceptions,Positioning,Vision,Penalties,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Preferred Foot,Body Type,Position
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""count""",15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,…,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,15298.0,"""15298""","""15298""","""15298"""
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0"""
"""mean""",214768.085175,25.05334,66.588704,71.570794,2.5891e6,10373.055301,1672.443391,1.119297,3.013139,2.541443,4.849784,5.75129,164.17754,60.015492,60.015492,60.015492,61.267224,60.946921,60.946921,60.946921,61.267224,61.223297,61.223297,61.223297,61.964897,60.469342,60.469342,60.469342,61.964897,59.720486,59.041116,59.041116,59.041116,59.720486,58.999346,57.782521,…,54.485096,49.961041,57.114002,62.86763,46.987253,60.82063,51.538959,46.709308,56.368087,63.418029,68.001046,68.061381,66.582494,62.365277,66.658844,59.839195,66.057262,67.494248,65.975291,51.720159,59.768336,50.629102,55.053406,55.75134,52.252386,60.533272,51.442738,52.101582,49.795725,10.511439,10.558374,10.557458,10.53641,10.514185,null,null,null
"""std""",28616.882947,4.542381,6.797187,6.099113,5.8318e6,23036.206191,196.70975,0.403074,0.638502,0.620319,1.678181,0.441699,14.78143,9.131929,9.131929,9.131929,9.869031,9.834071,9.834071,9.834071,9.869031,9.763917,9.763917,9.763917,9.247231,8.818658,8.818658,8.818658,9.247231,8.974868,10.077393,10.077393,10.077393,8.974868,9.525282,11.682026,…,14.09638,16.334868,11.601721,9.755208,14.814899,12.465115,15.156069,15.110173,12.342438,9.987623,11.735776,11.465788,12.291066,8.768651,12.144573,13.198908,11.512651,11.175881,12.546319,15.707688,14.406869,18.727823,14.627227,12.893842,12.518438,10.1669,17.179121,19.110227,19.116446,3.09629,3.076654,3.1474,3.085543,3.105028,null,null,null
"""min""",16.0,16.0,46.0,48.0,0.0,0.0,1000.0,1.0,1.0,2.0,0.0,5.1,110.0,33.0,33.0,33.0,27.0,29.0,29.0,29.0,27.0,29.0,29.0,29.0,29.0,32.0,32.0,32.0,29.0,32.0,30.0,30.0,30.0,32.0,31.0,27.0,…,11.0,10.0,15.0,20.0,10.0,14.0,11.0,10.0,19.0,25.0,20.0,25.0,23.0,21.0,22.0,14.0,28.0,27.0,25.0,11.0,13.0,10.0,11.0,12.0,12.0,22.0,10.0,10.0,10.0,1.0,1.0,1.0,1.0,1.0,"""Left""","""Lean""","""AM"""
"""25%""",200610.0,21.0,62.0,67.0,350000.0,1000.0,1532.0,1.0,3.0,2.0,4.0,5.11,154.0,54.0,54.0,54.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,55.0,56.0,55.0,55.0,55.0,56.0,54.0,52.0,52.0,52.0,54.0,52.0,48.0,…,45.0,36.0,49.0,57.0,35.0,55.0,40.0,34.0,49.0,58.0,62.0,62.0,59.0,56.0,60.0,52.0,59.0,61.0,59.0,40.0,50.0,35.0,47.0,47.0,43.0,54.0,38.0,35.0,32.0,8.0,8.0,8.0,8.0,8.0,null

In [52]:
df_limpio = df_sin_porteros.drop(["Age", "Overall", "Potential", "Value", "Wage", "Preferred Foot", "International Reputation"])
df_limpio

ID,Special,Weak Foot,Skill Moves,Work Rate,Height,Weight,LS,ST,RS,LW,LF,CF,RF,RW,LAM,CAM,RAM,LM,LCM,CM,RCM,RM,LWB,LDM,CDM,RDM,RWB,LB,LCB,CB,RCB,RB,Crossing,Finishing,HeadingAccuracy,ShortPassing,Volleys,Dribbling,Curve,FKAccuracy,LongPassing,BallControl,Acceleration,SprintSpeed,Agility,Reactions,Balance,ShotPower,Jumping,Stamina,Strength,LongShots,Aggression,Interceptions,Positioning,Vision,Penalties,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Body Type,Position
i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str
158023,2202,4,4,4,5.7,159,90,90,90,94,95,95,95,94,95,95,95,93,86,86,86,93,66,63,63,63,66,61,49,49,49,61,84,95,70,90,86,97,93,94,87,96,91,86,91,95,95,85,68,72,59,94,48,22,94,94,75,96,33,28,26,6,11,15,14,8,"""Lean""","""ST"""
20801,2228,4,5,6,6.2,183,94,94,94,92,93,93,93,92,91,91,91,91,84,84,84,91,68,64,64,64,68,64,56,56,56,64,84,94,89,81,87,88,81,76,77,94,89,91,87,96,70,95,95,88,79,93,63,29,95,82,85,95,28,31,23,7,11,15,14,11,"""Normal""","""ST"""
190871,2143,5,5,7,5.9,150,87,87,87,92,92,92,92,92,92,92,92,91,84,84,84,91,68,63,63,63,68,63,50,50,50,63,79,87,62,84,84,96,88,87,78,95,94,90,96,94,84,80,61,81,49,82,56,36,89,87,81,94,27,24,33,9,9,15,15,11,"""Lean""","""AM"""
192985,2281,5,4,8,5.11,154,85,85,85,90,90,90,90,90,91,91,91,91,90,90,90,91,80,80,80,80,80,76,69,69,69,76,93,82,55,92,82,86,85,83,91,91,78,76,79,91,77,91,63,90,75,91,76,61,87,94,79,88,68,58,51,15,13,5,10,13,"""Normal""","""MF"""
183277,2142,4,4,7,5.8,163,86,86,86,92,91,91,91,92,92,92,92,92,85,85,85,92,69,66,66,66,69,63,52,52,52,63,81,84,61,89,80,95,83,79,83,94,94,88,95,90,94,82,56,83,66,80,54,41,87,89,86,91,34,27,22,11,12,6,8,8,"""Normal""","""ST"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
243413,1098,3,2,4,6.1,168,34,34,34,31,32,32,32,31,30,30,30,32,32,32,32,32,41,40,40,40,41,44,48,48,48,44,22,23,45,25,27,21,21,27,27,32,52,52,39,43,48,39,74,39,52,16,44,45,20,31,38,43,44,47,53,9,10,9,11,13,"""Lean""","""DF"""
243165,1098,2,2,4,6.3,170,47,47,47,41,44,44,44,41,42,42,42,40,37,37,37,40,32,33,33,33,32,31,34,34,34,31,23,52,52,43,36,39,32,20,25,40,41,39,38,40,52,41,47,43,67,42,47,16,46,33,43,42,22,15,19,10,9,9,5,12,"""Normal""","""ST"""
241638,1189,3,2,4,5.8,148,47,47,47,47,48,48,48,47,46,46,46,46,40,40,40,46,36,32,32,32,36,35,30,30,30,35,25,40,46,38,38,45,38,27,28,44,70,69,50,47,58,45,60,55,32,45,32,15,48,43,55,41,32,13,11,6,5,10,6,13,"""Normal""","""ST"""
